# ⚡ Fast Downloader → Google Drive

Paste **any** link (direct file, magnet/torrent, or YouTube/video) and this notebook grabs it using Colab's fast server connection, then saves it straight to your Google Drive.

**How to use:**
1. Run each cell top to bottom (Shift+Enter)
2. When the Drive mount cell runs, click the link and authorize
3. In the last cell, paste your link(s) and hit enter



In [ ]:
# Install the tools we need
!apt-get install -y aria2 -q
!pip install -q yt-dlp

In [ ]:
# Mount your Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Where files get saved (change the folder name if you want)
import os

SAVE_FOLDER = "/content/drive/MyDrive/ColabDownloads"
os.makedirs(SAVE_FOLDER, exist_ok=True)
print(f"Files will be saved to: {SAVE_FOLDER}")

In [ ]:
# The smart download function — auto-detects link type & displays real-time progress & speed
import subprocess, re

def download_link(link, save_folder=SAVE_FOLDER):
    link = link.strip()
    if not link:
        return

    if link.startswith("magnet:"):
        print("🧲 Magnet/torrent link detected — downloading with aria2c...")
        cmd = ["aria2c", "--dir", save_folder, "--seed-time=0",
               "--bt-stop-timeout=120", "--summary-interval=1", "--download-result=full", link]

    elif re.search(r"(youtube\.com|youtu\.be)", link):
        print("🎥 YouTube/video link detected — downloading with yt-dlp...")
        cmd = ["yt-dlp", "--newline", "--progress", "-P", save_folder, link]

    else:
        print("📄 Direct file link detected — downloading with aria2c (16 connections)...")
        cmd = ["aria2c", "-x", "16", "-s", "16", "-k", "1M", "--dir", save_folder, "--summary-interval=1", "--download-result=full", link]

    result = subprocess.run(cmd)
    if result.returncode == 0:
        print(f"✅ Done! Check {save_folder} in your Drive.\n")
    else:
        print("❌ Something went wrong — double check the link.\n")

In [ ]:
# Run this cell, paste your link(s) (comma-separated for multiple), hit Enter
links = input("Paste your link(s): ").split(",")
for l in links:
    download_link(l)